# Bagging (BG) on MHAR Feature Set

**Purpose:** This notebook implements the bagging (BG) model of Breiman (1996) on the MHAR feature set for the EURO STOXX 50 index, following the tree-based methodology in Christensen, Siggaard, and Veliyev (2023). Bagging reduces the variance of a single regression tree by averaging predictions across many bootstrap-resampled trees. In the reference paper, BG is treated as an off-the-shelf benchmark: the training and validation samples are merged, the model is re-estimated in a rolling window, and the tree parameters are kept at the default Breiman-Cutler settings rather than tuned in the validation set. We mirror that implementation here to keep the exercise directly comparable with HAR, RF, and the regularized linear models.

## 1. Imports

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor


## 2. Data Loading

We use the same `MALL.csv` data source as in the other MHAR notebooks so that the tree-based models are evaluated on the exact same sample and information set.

In [ ]:
MALL = pd.read_csv("/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/MALL.csv")
MALL["Date"] = pd.to_datetime(MALL["Date"])
MALL = MALL.set_index("Date")

print(f"Sample period: {MALL.index[0].date()} to {MALL.index[-1].date()}")
print(f"Number of observations: {len(MALL)}")
display(MALL[["RVD", "RVW", "RVM", "RV_target"]].head())


## 3. Data Split

We keep the same 70/10/20 split as in the benchmark-model notebooks. As in the paper, BG does not require validation-set tuning, so the training and validation samples are merged into the rolling estimation window.

In [ ]:
n = len(MALL)
n_train = int(0.70 * n)
n_val = int(0.10 * n)
n_test = n - n_train - n_val

print(f"Training observations:   {n_train}")
print(f"Validation observations: {n_val}")
print(f"Test observations:       {n_test}")

print(f"Training period:   {MALL.index[0].date()} to {MALL.index[n_train - 1].date()}")
print(f"Validation period: {MALL.index[n_train].date()} to {MALL.index[n_train + n_val - 1].date()}")
print(f"Test period:       {MALL.index[n_train + n_val].date()} to {MALL.index[-1].date()}")


## 4. Feature Set Construction and Standardization

The paper reports standardizing all inputs prior to estimation. We therefore scale the MHAR regressors using training-sample moments, even though tree-based methods are less sensitive to scaling than linear models.

In [ ]:
y = MALL["RV_target"].values
X_MHAR = MALL[["RVD", "RVW", "RVM"]].values

X_train = X_MHAR[:n_train]
X_val = X_MHAR[n_train:n_train + n_val]
X_test = X_MHAR[n_train + n_val:]

y_train = y[:n_train]
y_val = y[n_train:n_train + n_val]
y_test = y[n_train + n_val:]

X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std = np.where(X_std == 0, 1.0, X_std)
X_scaled = (X_MHAR - X_mean) / X_std

print("Feature set: RVD, RVW, RVM")
print(f"\nFull dataset:   X={X_scaled.shape}, y={y.shape}")
print(f"Training set:   X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Test set:       X={X_test.shape}, y={y_test.shape}")
print("\nStandardization check on training sample:")
print(f"  Means: {X_scaled[:n_train].mean(axis=0).round(6)}")
print(f"  Std:   {X_scaled[:n_train].std(axis=0).round(6)}")


## 5. Bagging Specification

Following the paper, we keep BG at the Breiman-Cutler default-style settings instead of tuning hyperparameters in the validation sample. We use `500` bootstrap trees and set the terminal-node size to `5`, matching the paper's Appendix A.4 description.

In [ ]:
base_tree = DecisionTreeRegressor(min_samples_leaf=5, random_state=42)
bg_params = {
    "estimator": base_tree,
    "n_estimators": 500,
    "bootstrap": True,
    "random_state": 42,
    "n_jobs": -1
}

print("Bagging parameters:")
for key, value in bg_params.items():
    print(f"  {key}: {value}")


## 6. Rolling Window Estimation and Forecasting

As in the paper, BG is estimated using the merged training-plus-validation window and rolled through the out-of-sample period. Each day, the ensemble is re-estimated and used to produce a one-step-ahead forecast.

In [ ]:
window_size = n_train + n_val

forecasts_BG = []
dates_test = []

start_time = time.time()

for i in range(n_test):
    X_window = X_scaled[i:i + window_size]
    y_window = y[i:i + window_size]

    model_final = BaggingRegressor(**bg_params)
    model_final.fit(X_window, y_window)

    X_forecast = X_scaled[i + window_size].reshape(1, -1)
    forecast = model_final.predict(X_forecast)[0]
    forecast = max(forecast, y_window.min())

    forecasts_BG.append(forecast)
    dates_test.append(MALL.index[i + window_size])

    if (i + 1) % 100 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"  Day {i + 1:4d}/{n_test} | Date: {MALL.index[i + window_size].date()} | Elapsed: {elapsed:.1f} min")

total_elapsed = (time.time() - start_time) / 60
print(f"\nTotal estimation time: {total_elapsed:.1f} minutes")
print(f"Forecasts generated: {len(forecasts_BG)}")
print(f"Forecast period: {dates_test[0].date()} to {dates_test[-1].date()}")


## 7. Save Forecasts

We save the BG forecast series and summary error metrics in the same format as the rest of the MHAR model notebooks.

In [ ]:
forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/Forcasts/MHAR"
os.makedirs(forecast_path, exist_ok=True)

test_dates = MALL.index[n_train + n_val:]
results_BG = pd.DataFrame({"RV_actual": y_test, "RV_forecast": forecasts_BG}, index=test_dates)
errors_BG = results_BG["RV_actual"] - results_BG["RV_forecast"]
mse_BG = np.mean(errors_BG ** 2)

results_BG.to_csv(os.path.join(forecast_path, "forecasts_BG.csv"))
mse_summary_BG = pd.DataFrame({
    "Model": ["BG"],
    "MSE": [mse_BG],
    "RMSE": [np.sqrt(mse_BG)],
    "MAE": [errors_BG.abs().mean()],
    "Mean_Error": [errors_BG.mean()]
}).set_index("Model")
mse_summary_BG.to_csv(os.path.join(forecast_path, "mse_summary_BG.csv"))

print(f"Forecasts saved to: {forecast_path}")
print(f"Rows saved: {len(results_BG)}")
print(f"Bagging test-set MSE: {mse_BG:.6e}")


## 8. Out-of-Sample Performance Evaluation

We benchmark BG against HAR on the same out-of-sample period to see whether variance reduction through bootstrap aggregation improves forecast accuracy in this setting.

In [ ]:
forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/Forcasts/MHAR"
results_HAR = pd.read_csv(os.path.join(forecast_path, "forecasts_HAR.csv"), index_col=0, parse_dates=True)
mse_HAR = np.mean((results_HAR["RV_actual"] - results_HAR["RV_forecast"]) ** 2)
errors_HAR = results_HAR["RV_actual"] - results_HAR["RV_forecast"]
rel_mse_BG = mse_BG / mse_HAR

print(f"Out-of-sample performance ({test_dates[0].date()} to {test_dates[-1].date()}):")
print(f"\n{'Model':<10} {'MSE':>12} {'RMSE':>12} {'MAE':>12} {'Mean Error':>12} {'Rel. MSE':>10}")
print("-" * 70)
print(f"{'HAR':<10} {mse_HAR:>12.6e} {np.sqrt(mse_HAR):>12.6e} {errors_HAR.abs().mean():>12.6e} {errors_HAR.mean():>12.6e} {1.0:>10.4f}")
print(f"{'BG':<10} {mse_BG:>12.6e} {np.sqrt(mse_BG):>12.6e} {errors_BG.abs().mean():>12.6e} {errors_BG.mean():>12.6e} {rel_mse_BG:>10.4f}")
print("-" * 70)
print(f"\nBG vs HAR: {'outperforms' if rel_mse_BG < 1 else 'underperforms'} by {abs(1 - rel_mse_BG) * 100:.2f}%")


## 9. Forecast Visualization

As in the other notebooks, we convert realized variance into annualized volatility before plotting the actual and forecasted series.

In [ ]:
results_plot_BG = pd.DataFrame({
    "Actual": np.sqrt(results_BG["RV_actual"] * 252) * 100,
    "Forecast": np.sqrt(results_BG["RV_forecast"] * 252) * 100
}, index=results_BG.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_plot_BG["Actual"], label="Actual RV", color="black", linewidth=0.8)
ax.plot(results_plot_BG["Forecast"], label="BG Forecast", color="red", linewidth=0.8, linestyle="--")
ax.set_title("Bagging — One-Day-Ahead Realized Volatility Forecast\nEURO STOXX 50 (2016–2020)", fontsize=12)
ax.set_ylabel("Annualized Volatility (%)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSummary of forecast errors:")
print(f"Mean error:  {errors_BG.mean():.6e}")
print(f"RMSE:        {np.sqrt(mse_BG):.6e}")
print(f"MAE:         {errors_BG.abs().mean():.6e}")
